In [ ]:
import pandas as pd
import numpy as np
import math
from typing import List, Dict, Tuple

# Global constants
EDU_RANK = {'Bac': 0, 'Bachelor': 1, 'License': 2, 'Master': 3, 'PhD': 4}

# ---------- 1. Data Loading & Preparation ----------
def load_data() -> Tuple[pd.DataFrame, pd.DataFrame]:
    try:
        seekers_df = pd.read_csv('emplo.csv')[[
            'gender', 'birth_date', 'age', 'city', 'department',
            'sector', 'years_experience', 'highest_education',
            'contract_type', 'technical_skills', 'language_proficiency',
            'education_history', 'edu_value', 'salary'
        ]]
    except FileNotFoundError:
        seekers_df = pd.DataFrame([{
            'gender': 'Male', 'birth_date': '1990-05-15', 'age': 33,
            'city': 'Algiers', 'department': 'IT', 'sector': 'Technology',
            'years_experience': 5, 'highest_education': 'Master',
            'contract_type': 'Full-time', 'technical_skills': 'Python, SQL, Spark',
            'language_proficiency': 'English, French', 'education_history': 'University of Algiers',
            'edu_value': 3, 'salary': 70000
        }])

    jobs_df = pd.DataFrame([
        {'job_id': 'DEV-001', 'required_skills': ['HYSYS', 'Communication', 'Problem Solving'],
         'min_education': 'License', 'min_experience': 3, 'salary_offer': 75000,
         'location': 'Algiers', 'sector': 'Energy & Petroleum', 'contract_type': 'CDD'},
        {'job_id': 'DATA-002', 'required_skills': ['python', 'Problem Solving', 'Teamwork'],
         'min_education': 'Bachelor', 'min_experience': 2, 'salary_offer': 65000,
         'location': 'Oran', 'sector': 'Data Science', 'contract_type': 'CDD'}
    ])
    
    return seekers_df, jobs_df

# ---------- 2. Core Matching Functions ----------
def preprocess_data(seekers_df: pd.DataFrame, jobs_df: pd.DataFrame) -> Tuple:
    """Preprocess and normalize all data for matching"""
    # Education ranking
    seekers_df['edu_rank'] = seekers_df['highest_education'].map(EDU_RANK).fillna(-1).astype(int)
    jobs_df['edu_rank'] = jobs_df['min_education'].map(EDU_RANK).fillna(-1).astype(int)

    # Skills processing
    seekers_df['technical_skills'] = seekers_df['technical_skills'].fillna('').str.lower()
    seeker_skills = [set(s.split(', ')) for s in seekers_df['technical_skills']]
    job_skills = [set(map(str.lower, req)) for req in jobs_df['required_skills']]

    # Sector and contract type normalization
    all_sectors = pd.Categorical(seekers_df['sector'].tolist() + jobs_df['sector'].tolist())
    seek_sector_codes = pd.Categorical(seekers_df['sector'], categories=all_sectors.categories).codes
    job_sector_codes = pd.Categorical(jobs_df['sector'], categories=all_sectors.categories).codes

    contract_types = pd.Categorical(seekers_df['contract_type'].tolist() + jobs_df['contract_type'].tolist())
    seek_cont_codes = pd.Categorical(seekers_df['contract_type'], categories=contract_types.categories).codes
    job_cont_codes = pd.Categorical(jobs_df['contract_type'], categories=contract_types.categories).codes

    return seekers_df, jobs_df, seeker_skills, job_skills, seek_sector_codes, job_sector_codes, seek_cont_codes, job_cont_codes

def calculate_features(seekers_df, jobs_df, seeker_skills, job_skills, seek_sector_codes, job_sector_codes, seek_cont_codes, job_cont_codes) -> np.ndarray:
    """Calculate all feature scores between seekers and jobs"""
    N_seek = len(seekers_df)
    N_job = len(jobs_df)
    F_raw = np.zeros((N_seek, N_job, 7), dtype=np.float32)

    # Skill TF-IDF calculation
    all_skills = [skill for skills in seekers_df['technical_skills'] for skill in skills.split(', ') if skill]
    skill_counts = pd.Series(all_skills).value_counts()
    total_docs = N_seek + N_job
    skill_idf = {s: np.log((total_docs + 1)/(cnt + 1)) + 1 for s, cnt in skill_counts.items()}

    # Salary normalization
    all_salaries = np.concatenate([seekers_df['salary'].values, jobs_df['salary_offer'].values])
    max_sal = np.percentile(all_salaries, 95)
    min_sal = np.percentile(all_salaries, 5)

    for i in range(N_seek):
        for j in range(N_job):
            # Skill match (40%)
            common_skills = seeker_skills[i] & job_skills[j]
            missing_skills = job_skills[j] - seeker_skills[i]
            penalty = 1 - (len(missing_skills) / len(job_skills[j]))
            match_score = sum(skill_idf.get(s, 0) for s in common_skills)
            total_score = sum(skill_idf.get(s, 0) for s in job_skills[j])
            F_raw[i,j,0] = penalty * (match_score / total_score if total_score > 0 else 0)

            # Experience (15%)
            F_raw[i,j,1] = min(seekers_df.at[i,'years_experience'] / max(jobs_df.at[j,'min_experience'], 1), 1.5)

            # Salary (15%)
            salary_diff = abs(jobs_df.at[j,'salary_offer'] - seekers_df.at[i,'salary'])
            F_raw[i,j,2] = 1 - np.log1p(salary_diff) / np.log1p(max_sal - min_sal)

            # Education (10%)
            F_raw[i,j,3] = seekers_df.at[i,'edu_rank'] / max(EDU_RANK.values())

            # Sector (10%)
            F_raw[i,j,4] = (seek_sector_codes[i] == job_sector_codes[j])

            # Contract (5%)
            F_raw[i,j,5] = (seek_cont_codes[i] == job_cont_codes[j])

            # Education value (5%)
            F_raw[i,j,6] = seekers_df.at[i,'edu_value'] / 20

    # Min-Max Normalization
    feature_mins = F_raw.min(axis=(0,1))
    feature_maxs = F_raw.max(axis=(0,1))
    F = (F_raw - feature_mins) / (feature_maxs - feature_mins + 1e-8)
    
    return F

# ---------- 3. Genetic Algorithm Implementation ----------
def initialize_population(pop_size: int, N_seek: int, valid_jobs: np.ndarray) -> np.ndarray:
    """Initialize population with valid job assignments"""
    pop = np.empty((pop_size, N_seek), dtype=int)
    for i in range(N_seek):
        choices = np.append(np.where(valid_jobs[i])[0], -1)  # -1 means no job
        pop[:,i] = np.random.choice(choices, pop_size)
    return pop

def calculate_fitness(pop: np.ndarray, F: np.ndarray, weights: np.ndarray) -> np.ndarray:
    """Calculate fitness scores for population"""
    scores = np.zeros(pop.shape[0])
    for k in range(pop.shape[0]):
        for i in range(pop.shape[1]):
            j = pop[k,i]
            if j >= 0:  # Only count if assigned to a valid job
                scores[k] += F[i,j].dot(weights)
        # Penalize unmatched candidates
        scores[k] -= 0.1 * np.sum(pop[k] == -1)
    return scores

def tournament_selection(pop: np.ndarray, fitness: np.ndarray, tournament_size: int = 3) -> np.ndarray:
    """Select parents using tournament selection"""
    selected = np.empty_like(pop)
    for i in range(pop.shape[0]):
        contenders = np.random.choice(len(fitness), tournament_size, replace=False)
        winner = pop[contenders[np.argmax(fitness[contenders])]]
        selected[i] = winner
    return selected

def crossover(parent1: np.ndarray, parent2: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Perform crossover between two parents"""
    if len(parent1) <= 1:
        return parent1.copy(), parent2.copy()
    cut = np.random.randint(1, len(parent1))
    child1 = np.concatenate([parent1[:cut], parent2[cut:]])
    child2 = np.concatenate([parent2[:cut], parent1[cut:]])
    return child1, child2

def mutate(chromosome: np.ndarray, mutation_rate: float, valid_jobs: np.ndarray) -> np.ndarray:
    """Mutate chromosome with given rate"""
    for i in range(len(chromosome)):
        if np.random.rand() < mutation_rate:
            choices = np.append(np.where(valid_jobs[i])[0], -1)
            chromosome[i] = np.random.choice(choices)
    return chromosome

def run_genetic_algorithm(F: np.ndarray, weights: np.ndarray, valid_jobs: np.ndarray,
                         pop_size: int = 50, generations: int = 100,
                         mutation_rate: float = 0.1, elite_size: int = 2) -> Tuple[np.ndarray, float]:
    """Run the complete genetic algorithm"""
    N_seek = F.shape[0]
    pop = initialize_population(pop_size, N_seek, valid_jobs)
    best_score = -np.inf
    best_solution = None
    
    for gen in range(generations):
        # Evaluate fitness
        fitness = calculate_fitness(pop, F, weights)
        
        # Track best solution
        current_best = np.argmax(fitness)
        if fitness[current_best] > best_score:
            best_score = fitness[current_best]
            best_solution = pop[current_best].copy()
        
        # Selection
        selected = tournament_selection(pop, fitness)
        
        # Crossover
        new_pop = []
        for i in range(0, pop_size - elite_size, 2):
            p1, p2 = selected[i], selected[i+1]
            c1, c2 = crossover(p1, p2)
            new_pop.extend([c1, c2])
        
        # Elitism
        elites = pop[np.argsort(fitness)[-elite_size:]]
        new_pop.extend(elites)
        
        # Mutation
        for i in range(len(new_pop)):
            new_pop[i] = mutate(new_pop[i], mutation_rate, valid_jobs)
        
        pop = np.array(new_pop)[:pop_size]  # Ensure population size stays constant
        
        print(f"Generation {gen+1}/{generations} | Best Score: {best_score:.2f}", end='\r')
    
    print()  # New line after progress updates
    return best_solution, best_score

# ---------- 4. Main Execution ----------
if __name__ == '__main__':
    # Load and preprocess data
    seekers_df, jobs_df = load_data()
    seekers_df, jobs_df, seeker_skills, job_skills, seek_sector_codes, job_sector_codes, seek_cont_codes, job_cont_codes = preprocess_data(seekers_df, jobs_df)
    
    # Calculate feature matrix
    F = calculate_features(seekers_df, jobs_df, seeker_skills, job_skills, 
                         seek_sector_codes, job_sector_codes, seek_cont_codes, job_cont_codes)
    
    # Define weights for each feature
    WEIGHTS = np.array([0.40, 0.15, 0.15, 0.10, 0.10, 0.05, 0.05], dtype=np.float32)
    
    # Create valid jobs matrix
    valid_jobs = (
    (seekers_df['edu_rank'].values[:, None] >= jobs_df['edu_rank'].values[None, :]) &
    (seekers_df['years_experience'].values[:, None] >= jobs_df['min_experience'].values[None, :]) &
    np.array([
        [len(seeker_skills[i] & job_skills[j]) >= max(1, len(job_skills[j]) // 2) for j in range(len(jobs_df))]
        for i in range(len(seekers_df))
    ])
    )

    
    # Run genetic algorithm
    best_solution, best_score = run_genetic_algorithm(F, WEIGHTS, valid_jobs)
    
    # Display results
    print("\nBest Solution Score:", best_score)
    print("\nOptimal Matches:")
    for i, job_idx in enumerate(best_solution):
        if job_idx >= 0:  # Only show actual matches
            seeker = seekers_df.iloc[i]
            job = jobs_df.iloc[job_idx]
            
            # Calculate match details
            common_skills = seeker_skills[i] & job_skills[job_idx]
            missing_skills = job_skills[job_idx] - seeker_skills[i]
            sector_match = seeker['sector'] == job['sector']
            contract_match = seeker['contract_type'] == job['contract_type']
            
            print(f"\nSeeker {i} → Job {job['job_id']}")
            print(f"  Skills: {len(common_skills)}/{len(job_skills[job_idx])} matched")
            if missing_skills:
                print(f"  Missing Skills: {', '.join(missing_skills)}")
            print(f"  Sector: {'Match' if sector_match else f'Mismatch (Seeker: {seeker["sector"]}, Job: {job["sector"]})'}")
            print(f"  Contract: {'Match' if contract_match else f'Mismatch (Seeker: {seeker["contract_type"]}, Job: {job["contract_type"]})'}")
            print(f"  Experience: {seeker['years_experience']}y (Req: {job['min_experience']}y)")
            print(f"  Education: {seeker['highest_education']} (Req: {job['min_education']})")
            print(f"  Salary: Seeker ${seeker['salary']:,} vs Job Offer ${job['salary_offer']:,}")
    
    # Show top candidates per job
    print("\nTop Candidates per Job:")
    for j in range(len(jobs_df)):
        job = jobs_df.iloc[j]
        scores = F[:,j].dot(WEIGHTS)
        valid = valid_jobs[:,j]
        ranked = sorted([(i, scores[i]) for i in np.where(valid)[0]], key=lambda x: -x[1])[:5]
        
        print(f"\nJob {job['job_id']} ({job['sector']}):")
        for rank, (i, score) in enumerate(ranked, 1):
            seeker = seekers_df.iloc[i]
            common_skills = seeker_skills[i] & job_skills[j]
            print(f"{rank}. [Score: {score:.2%}] {seeker['technical_skills']}")
            print(f"   Sector: {seeker['sector']} | Exp: {seeker['years_experience']}y | Edu: {seeker['highest_education']}")
            print(f"   Matching Skills: {', '.join(common_skills)}")







Generation 100/100 | Best Score: 145.06

Best Solution Score: 145.05658100843425

Optimal Matches:

Seeker 0 → Job DATA-002
  Skills: 1/3 matched
  Missing Skills: problem solving, python
  Sector: Mismatch (Seeker: Energy & Petroleum, Job: Data Science)
  Contract: Match
  Experience: 9y (Req: 2y)
  Education: License (Req: Bachelor)
  Salary: Seeker $100,800.0 vs Job Offer $65,000

Seeker 1 → Job DATA-002
  Skills: 1/3 matched
  Missing Skills: teamwork, python
  Sector: Mismatch (Seeker: E-Commerce, Job: Data Science)
  Contract: Mismatch (Seeker: Stage, Job: CDD)
  Experience: 6y (Req: 2y)
  Education: Master (Req: Bachelor)
  Salary: Seeker $96,200.0 vs Job Offer $65,000

Seeker 5 → Job DEV-001
  Skills: 2/3 matched
  Missing Skills: hysys
  Sector: Mismatch (Seeker: Aerospace & Defense, Job: Energy & Petroleum)
  Contract: Mismatch (Seeker: Stage, Job: CDD)
  Experience: 36y (Req: 3y)
  Education: Master (Req: License)
  Salary: Seeker $134,800.0 vs Job Offer $75,000

Seeker 6 → 

In [ ]:
```python
import pandas as pd
import numpy as np
from typing import Tuple

# ---------- Constants ----------
WEIGHTS = np.array([0.40, 0.15, 0.15, 0.10, 0.10, 0.05, 0.05], dtype=np.float32)

# ---------- 1. Data Loading & Preparation ----------
def load_data() -> Tuple[pd.DataFrame, pd.DataFrame]:
    try:
        seekers_df = pd.read_csv('emplo.csv')[[
            'gender', 'birth_date', 'age', 'city', 'department',
            'sector', 'years_experience', 'highest_education',
            'contract_type', 'technical_skills', 'language_proficiency',
            'education_history', 'edu_value', 'salary'
        ]]
    except FileNotFoundError:
        seekers_df = pd.DataFrame([{
            'gender': 'Male', 'birth_date': '1990-05-15', 'age': 33,
            'city': 'Algiers', 'department': 'IT', 'sector': 'Technology',
            'years_experience': 5, 'highest_education': 'Master',
            'contract_type': 'Full-time', 'technical_skills': 'Python, SQL, Spark',
            'language_proficiency': 'English, French', 'education_history': 'University of Algiers',
            'edu_value': 3, 'salary': 70000
        }])

    jobs_df = pd.DataFrame([
        {'job_id': 'DEV-001', 'required_skills': ['HYSYS', 'Communication', 'Problem Solving'],
         'min_education_value': 2, 'min_experience': 3, 'salary_offer': 75000,
         'location': 'Algiers', 'sector': 'Energy & Petroleum', 'contract_type': 'CDD'},
        {'job_id': 'DATA-002', 'required_skills': ['python', 'Problem Solving', 'Teamwork'],
         'min_education_value': 1, 'min_experience': 2, 'salary_offer': 65000,
         'location': 'Oran', 'sector': 'Data Science', 'contract_type': 'CDD'}
    ])
    return seekers_df, jobs_df

# ---------- 2. Preprocessing ----------
def preprocess_data(seekers_df: pd.DataFrame, jobs_df: pd.DataFrame) -> Tuple:
    # Lowercase skills
    seekers_df['technical_skills'] = seekers_df['technical_skills'].fillna('').str.lower()
    seeker_skills = [set(s.split(', ')) for s in seekers_df['technical_skills']]
    job_skills = [set(map(str.lower, req)) for req in jobs_df['required_skills']]

    # Sector & contract normalization
    all_sectors = pd.Categorical(seekers_df['sector'].tolist() + jobs_df['sector'].tolist())
    seek_sector_codes = pd.Categorical(seekers_df['sector'], categories=all_sectors.categories).codes
    job_sector_codes = pd.Categorical(jobs_df['sector'], categories=all_sectors.categories).codes

    contract_types = pd.Categorical(seekers_df['contract_type'].tolist() + jobs_df['contract_type'].tolist())
    seek_cont_codes = pd.Categorical(seekers_df['contract_type'], categories=contract_types.categories).codes
    job_cont_codes = pd.Categorical(jobs_df['contract_type'], categories=contract_types.categories).codes

    return seekers_df, jobs_df, seeker_skills, job_skills, seek_sector_codes, job_sector_codes, seek_cont_codes, job_cont_codes

# ---------- 3. Feature Calculation ----------
def calculate_features(seekers_df, jobs_df, seeker_skills, job_skills, 
                       seek_sector_codes, job_sector_codes, seek_cont_codes, job_cont_codes) -> np.ndarray:
    N_seek, N_job = len(seekers_df), len(jobs_df)
    F_raw = np.zeros((N_seek, N_job, 7), dtype=np.float32)

    # TF-IDF for skills
    all_skills = [skill for skills in seekers_df['technical_skills'] for skill in skills.split(', ') if skill]
    skill_counts = pd.Series(all_skills).value_counts()
    total_docs = N_seek + N_job
    skill_idf = {s: np.log((total_docs + 1)/(cnt + 1)) + 1 for s, cnt in skill_counts.items()}

    # Salary bounds
    all_salaries = np.concatenate([seekers_df['salary'].values, jobs_df['salary_offer'].values])
    max_sal = np.percentile(all_salaries, 95)
    min_sal = np.percentile(all_salaries, 5)

    max_edu = seekers_df['edu_value'].max()

    for i in range(N_seek):
        for j in range(N_job):
            # Skill match
            common = seeker_skills[i] & job_skills[j]
            missing = job_skills[j] - seeker_skills[i]
            penalty = 1 - (len(missing)/len(job_skills[j]))
            match = sum(skill_idf.get(s,0) for s in common)
            total = sum(skill_idf.get(s,0) for s in job_skills[j])
            F_raw[i,j,0] = penalty * (match/total if total>0 else 0)

            # Experience
            F_raw[i,j,1] = min(seekers_df.at[i,'years_experience']/max(jobs_df.at[j,'min_experience'],1),1.5)

            # Salary
            diff = abs(jobs_df.at[j,'salary_offer']-seekers_df.at[i,'salary'])
            F_raw[i,j,2] = 1 - np.log1p(diff)/np.log1p(max_sal-min_sal)

            # Education (using edu_value)
            F_raw[i,j,3] = seekers_df.at[i,'edu_value']/max_edu

            # Sector
            F_raw[i,j,4] = (seek_sector_codes[i]==job_sector_codes[j])

            # Contract
            F_raw[i,j,5] = (seek_cont_codes[i]==job_cont_codes[j])

            # Edu value normalized
            F_raw[i,j,6] = seekers_df.at[i,'edu_value']/20

    # Normalize 0-1
    mins, maxs = F_raw.min((0,1)), F_raw.max((0,1))
    return (F_raw-mins)/(maxs-mins+1e-8)

# ---------- 4. Valid Jobs Mask ----------
def build_valid_jobs(seekers_df, seeker_skills, job_skills, jobs_df) -> np.ndarray:
    return (
        (seekers_df['edu_value'].values[:,None] >= jobs_df['min_education_value'].values[None,:]) &
        (seekers_df['years_experience'].values[:,None] >= jobs_df['min_experience'].values[None,:]) &
        np.array([
            [len(seeker_skills[i]&job_skills[j]) >= max(1,len(job_skills[j])//2)
             for j in range(len(jobs_df))]
            for i in range(len(seekers_df))
        ])
    )

# ---------- 5. Reverse Search Function ----------
def find_seekers_for_job(job_id: str,
                          seekers_df: pd.DataFrame,
                          jobs_df: pd.DataFrame,
                          F: np.ndarray,
                          valid_jobs: np.ndarray,
                          top_k: int = 5) -> pd.DataFrame:
    jidx = jobs_df.index[jobs_df['job_id']==job_id][0]
    scores = F[:,jidx].dot(WEIGHTS)
    mask = valid_jobs[:,jidx]
    idxs = np.where(mask)[0]

    df = pd.DataFrame({'idx':idxs, 'score':scores[idxs]})
    df = df.sort_values('score',ascending=False).head(top_k)

    top = seekers_df.reset_index(drop=True).loc[df['idx']].copy()
    top['match_score'] = df['score'].values
    return top

# ---------- Main ----------
if __name__ == '__main__':
    seekers_df, jobs_df = load_data()

    seekers_df, jobs_df, seeker_skills, job_skills, \
        seek_sector_codes, job_sector_codes, seek_cont_codes, job_cont_codes = preprocess_data(seekers_df, jobs_df)

    F = calculate_features(seekers_df, jobs_df, seeker_skills, job_skills,
                           seek_sector_codes, job_sector_codes, seek_cont_codes, job_cont_codes)

    valid_jobs = build_valid_jobs(seekers_df, seeker_skills, job_skills, jobs_df)

    top_seekers = find_seekers_for_job('DATA-002', seekers_df, jobs_df, F, valid_jobs)
    print("Top Seekers for DATA-002:")
    print(top_seekers[['gender','age','city','years_experience','highest_education','match_score']])


In [ ]:
import pandas as pd
import numpy as np
from typing import Tuple, List

# ---------- Constants ----------
WEIGHTS = np.array([0.40, 0.15, 0.15, 0.10, 0.10, 0.05, 0.05], dtype=np.float32)

# ---------- 1. Data Loading & Preparation ----------

def load_jobs(filepath: str) -> pd.DataFrame:
    """
    Load the jobs dataset from CSV with expected columns:
    ['job_title','department','date_posted','application_deadline','valid_through',
     'sector','job_location','education_requirement','education_years','type_of_contract',
     'available_posts','benefits','technical_skills','experience_min_req',
     'experience_max_req','edu_value','language_proficiency',
     'education_requirement_std','salary']
    """
    jobs_df = pd.read_csv(filepath)
    # Normalize skill lists
    jobs_df['technical_skills'] = jobs_df['technical_skills'].fillna('').apply(lambda s: [skill.strip().lower() for skill in s.split(',') if skill])
    return jobs_df

# ---------- 2. Preprocessing & Feature Helpers ----------

def preprocess_job_features(jobs_df: pd.DataFrame) -> Tuple[List[set], np.ndarray, np.ndarray, np.ndarray]:
    # Skill sets
    job_skills = [set(skills) for skills in jobs_df['technical_skills']]
    # Sector codes
    sector_codes = pd.Categorical(jobs_df['sector']).codes
    # Contract codes
    contract_codes = pd.Categorical(jobs_df['type_of_contract']).codes
    return job_skills, sector_codes, contract_codes

# TF-IDF IDF values for job skills

def compute_skill_idf(jobs_skills: List[set], seeker_skills: set) -> dict:
    docs = jobs_skills + [seeker_skills]
    all_terms = [t for doc in docs for t in doc]
    counts = pd.Series(all_terms).value_counts()
    N = len(docs)
    return {term: np.log((N+1)/(cnt+1)) + 1 for term, cnt in counts.items()}

# ---------- 3. Feature Calculation for Single Seeker ----------

def calculate_seeker_job_features(seeker: dict,
                                   jobs_df: pd.DataFrame,
                                   job_skills: List[set],
                                   sector_codes: np.ndarray,
                                   contract_codes: np.ndarray) -> np.ndarray:
    """
    Build feature matrix F_raw (1 x N_jobs x 7) and normalize.
    """
    N_job = len(jobs_df)
    F_raw = np.zeros((1, N_job, 7), dtype=np.float32)

    # Prepare seeker skill set
    seeker_set = set([s.strip().lower() for s in seeker['technical_skills'].split(',') if s])

    # IDF
    skill_idf = compute_skill_idf(job_skills, seeker_set)

    # Salary bounds
    all_salaries = np.concatenate([jobs_df['salary'].values, [seeker['salary']]])
    max_sal, min_sal = np.percentile(all_salaries, 95), np.percentile(all_salaries, 5)

    # Sector & contract for seeker
    all_sectors = list(pd.Categorical(jobs_df['sector']).categories)
    try:
        seeker_sector_code = all_sectors.index(seeker['sector'])
    except ValueError:
        seeker_sector_code = -1

    all_contracts = list(pd.Categorical(jobs_df['type_of_contract']).categories)
    try:
        seeker_contract_code = all_contracts.index(seeker['contract_type'])
    except ValueError:
        seeker_contract_code = -1

    max_edu = max(jobs_df['edu_value'].max(), seeker['edu_value'])

    for j in range(N_job):
        # Skills
        common = seeker_set & job_skills[j]
        missing = job_skills[j] - seeker_set
        penalty = 1 - (len(missing)/max(len(job_skills[j]),1))
        match = sum(skill_idf.get(s, 0) for s in common)
        total = sum(skill_idf.get(s, 0) for s in job_skills[j])
        F_raw[0, j, 0] = penalty * (match/total if total>0 else 0)

        # Experience
        F_raw[0, j, 1] = min(seeker['years_experience']/max(jobs_df.at[j,'experience_min_req'],1),
                             jobs_df.at[j,'experience_max_req']/max(jobs_df.at[j,'experience_min_req'],1))

        # Salary
        diff = abs(jobs_df.at[j,'salary'] - seeker['salary'])
        F_raw[0, j, 2] = 1 - np.log1p(diff)/np.log1p(max_sal-min_sal)

        # Education
        F_raw[0, j, 3] = seeker['edu_value']/max_edu

        # Sector
        F_raw[0, j, 4] = int(seeker_sector_code == sector_codes[j])

        # Contract
        F_raw[0, j, 5] = int(seeker_contract_code == contract_codes[j])

        # Language proficiency (as proxy for edu value feature)
        F_raw[0, j, 6] = seeker['edu_value']/20

    # Min-max normalize features
    mins = F_raw.min(axis=(0,1))
    maxs = F_raw.max(axis=(0,1))
    return (F_raw - mins)/(maxs - mins + 1e-8)

# ---------- 4. Valid Jobs Mask for Seeker ----------

def build_valid_job_mask(seeker: dict, jobs_df: pd.DataFrame, job_skills: List[set]) -> np.ndarray:
    N_job = len(jobs_df)
    mask = np.zeros((1, N_job), dtype=bool)
    seeker_set = set([s.strip().lower() for s in seeker['technical_skills'].split(',') if s])
    for j in range(N_job):
        edu_ok = seeker['edu_value'] >= jobs_df.at[j,'edu_value']
        exp_ok = (seeker['years_experience'] >= jobs_df.at[j,'experience_min_req'] and
                  seeker['years_experience'] <= jobs_df.at[j,'experience_max_req'])
        skills_ok = len(seeker_set & job_skills[j]) >= max(1, len(job_skills[j])//2)
        mask[0,j] = (edu_ok and exp_ok and skills_ok)
    return mask

# ---------- 5. Reverse Search: Top Jobs for Seeker ----------

def find_jobs_for_seeker(seeker: dict,
                          jobs_df: pd.DataFrame,
                          F: np.ndarray,
                          valid_mask: np.ndarray,
                          top_k: int = 5) -> pd.DataFrame:
    scores = F[0].dot(WEIGHTS)
    idxs = np.where(valid_mask[0])[0]

    df = pd.DataFrame({'job_index': idxs, 'score': scores[idxs]})
    df = df.sort_values('score', ascending=False).head(top_k)

    top_jobs = jobs_df.reset_index(drop=True).loc[df['job_index']].copy()
    top_jobs['match_score'] = df['score'].values
    return top_jobs

# ---------- 6. Main Interaction ----------

if __name__ == '__main__':
    # Load jobs
    jobs_df = load_jobs('Users\\Hassane\\Documents\\Ai-project\\data\\emplo.csv')
    # Preprocess jobs
    job_skills, sector_codes, contract_codes = preprocess_job_features(jobs_df)

    # Example: input seeker information (could be replaced with real user input)
    seeker_input = {
        'gender': 'Female',
        'birth_date': '1992-08-10',
        'age': 32,
        'city': 'Oran',
        'department': 'Data Science',
        'sector': 'Data Science',
        'years_experience': 4,
        'highest_education': 'Master',
        'contract_type': 'CDD',
        'technical_skills': 'python, SQL, Machine Learning',
        'language_proficiency': 'English, French',
        'education_history': 'University of Oran',
        'edu_value': 3,
        'salary': 60000
    }

    # Calculate features & mask
    F = calculate_seeker_job_features(seeker_input, jobs_df, job_skills, sector_codes, contract_codes)
    valid_mask = build_valid_job_mask(seeker_input, jobs_df, job_skills)

    # Find top job offers
    recommended = find_jobs_for_seeker(seeker_input, jobs_df, F, valid_mask, top_k=5)
    print("Top 5 job offers for the candidate:")
    print(recommended[['job_title','department','sector','job_location','salary','match_score']])

FileNotFoundError: [Errno 2] No such file or directory: 'Users\\Hassane\\Documents\\Ai-project\\data\\emplo.csv'